In [1]:
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap


# =========================================================
# CONFIG
# =========================================================

MODEL_PATH = "two_tower_full_bundle.pt"
DATA_PATH = "my_data_with_wagon_id.csv"
OUTPUT_PATH = "wagons_data.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

TARGET_COLS = ["y_14d", "y_90d"]
ID_CANDIDATES = ["wagon_id"]
DATE_CANDIDATES = ["День отправки", "date", "dispatch_date"]

# SHAP для всех вагонов
BACKGROUND_SIZE = 80
EXPLAIN_MAX_ROWS = None


# =========================================================
# UTILS
# =========================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


def safe_float(x):
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None


def safe_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.integer, int)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return float(x)
    return str(x)


def risk_level(r14, r90, thr14, thr90):
    high = (r14 >= thr14) or (r90 >= thr90)
    med = (r14 >= max(0.35, thr14 - 0.10)) or (r90 >= max(0.35, thr90 - 0.10))
    if high:
        return "High"
    if med:
        return "Medium"
    return "Low"


def first_existing(cols, frame):
    for c in cols:
        if c in frame.columns:
            return c
    return None


def is_russia_exp_station(station_name):
    if pd.isna(station_name):
        return False
    s = str(station_name).lower().strip()
    return "(эксп)" in s or "(эксп.)" in s


# =========================================================
# PREPROCESSING CLASSES
# =========================================================

class CategoryEncoder:
    def __init__(self):
        self.maps = {}
        self.cardinalities = {}

    def transform(self, df, cat_cols):
        out = pd.DataFrame(index=df.index)
        for col in cat_cols:
            vocab = self.maps[col]
            out[col] = (
                df[col]
                .fillna("__MISSING__")
                .astype(str)
                .map(lambda x: vocab.get(x, 0))
                .astype("int64")
            )
        return out


class NumericScaler:
    def __init__(self):
        self.cols = None
        self.means = None
        self.stds = None

    def transform(self, df):
        if not self.cols:
            return pd.DataFrame(index=df.index)

        x = df[self.cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        x = (x - self.means) / self.stds
        return x.astype("float32")


# =========================================================
# MODEL
# =========================================================

def emb_dim(cardinality):
    return min(64, max(4, int(math.sqrt(cardinality)) + 1))


class CategoricalEmbeddingBlock(nn.Module):
    def __init__(self, cardinalities):
        super().__init__()
        self.num_features = len(cardinalities)
        self.emb_layers = nn.ModuleList([
            nn.Embedding(card, emb_dim(card))
            for card in cardinalities
        ])
        self.output_dim = sum(emb_dim(card) for card in cardinalities)

    def forward(self, x):
        if self.num_features == 0:
            return None
        embs = []
        for i, emb in enumerate(self.emb_layers):
            embs.append(emb(x[:, i]))
        return torch.cat(embs, dim=1)


class Tower(nn.Module):
    def __init__(self, cat_cardinalities, num_features, tower_hidden=(256, 128), dropout=0.20):
        super().__init__()

        self.has_cat = len(cat_cardinalities) > 0
        self.has_num = num_features > 0

        if self.has_cat:
            self.cat_block = CategoricalEmbeddingBlock(cat_cardinalities)
            cat_dim = self.cat_block.output_dim
        else:
            cat_dim = 0

        if self.has_num:
            self.num_bn = nn.BatchNorm1d(num_features)
            num_dim = num_features
        else:
            num_dim = 0

        input_dim = cat_dim + num_dim
        if input_dim == 0:
            raise ValueError("Tower has no input features")

        layers = []
        prev = input_dim
        for h in tower_hidden:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev = h

        self.mlp = nn.Sequential(*layers)
        self.output_dim = prev

    def forward(self, x_cat, x_num):
        parts = []

        if self.has_cat:
            parts.append(self.cat_block(x_cat))
        if self.has_num:
            parts.append(self.num_bn(x_num))

        x = torch.cat(parts, dim=1)
        return self.mlp(x)


class TwoTowerNetV2(nn.Module):
    def __init__(
        self,
        wagon_cat_cardinalities,
        wagon_num_features,
        context_cat_cardinalities,
        context_num_features,
        wagon_hidden=(256, 128),
        context_hidden=(128, 64),
        fusion_hidden=(256, 128, 64),
        dropout=0.20
    ):
        super().__init__()

        self.wagon_tower = Tower(
            wagon_cat_cardinalities,
            wagon_num_features,
            tower_hidden=wagon_hidden,
            dropout=dropout
        )

        self.context_tower = Tower(
            context_cat_cardinalities,
            context_num_features,
            tower_hidden=context_hidden,
            dropout=dropout
        )

        z_dim = self.wagon_tower.output_dim
        c_dim = self.context_tower.output_dim

        if z_dim != c_dim:
            common_dim = min(z_dim, c_dim)
            self.align_w = nn.Linear(z_dim, common_dim)
            self.align_c = nn.Linear(c_dim, common_dim)
            fusion_input_dim = common_dim * 4
        else:
            self.align_w = None
            self.align_c = None
            fusion_input_dim = z_dim * 4

        layers = []
        prev = fusion_input_dim
        for h in fusion_hidden:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev = h

        self.fusion = nn.Sequential(*layers)
        self.head_14 = nn.Linear(prev, 1)
        self.head_90 = nn.Linear(prev, 1)

    def forward(self, wagon_cat, wagon_num, context_cat, context_num):
        z_w = self.wagon_tower(wagon_cat, wagon_num)
        z_c = self.context_tower(context_cat, context_num)

        if self.align_w is not None:
            z_w = self.align_w(z_w)
            z_c = self.align_c(z_c)

        z_abs = torch.abs(z_w - z_c)
        z_mul = z_w * z_c
        z = torch.cat([z_w, z_c, z_abs, z_mul], dim=1)
        z = self.fusion(z)

        logit_14 = self.head_14(z)
        logit_90 = self.head_90(z)
        return logit_14, logit_90


# =========================================================
# LOAD MODEL BUNDLE
# =========================================================

bundle = torch.load(MODEL_PATH, map_location=DEVICE)

wagon_cat_cols = bundle["wagon_cat_cols"]
wagon_num_cols = bundle["wagon_num_cols"]
context_cat_cols = bundle["context_cat_cols"]
context_num_cols = bundle["context_num_cols"]

wagon_cat_cardinalities = bundle["wagon_cat_cardinalities"]
context_cat_cardinalities = bundle["context_cat_cardinalities"]

best_threshold_14 = float(bundle.get("best_threshold_14", 0.5))
best_threshold_90 = float(bundle.get("best_threshold_90", 0.5))

wagon_cat_encoder = CategoryEncoder()
wagon_cat_encoder.maps = bundle["wagon_cat_maps"]
wagon_cat_encoder.cardinalities = {
    col: len(vocab) for col, vocab in wagon_cat_encoder.maps.items()
}

context_cat_encoder = CategoryEncoder()
context_cat_encoder.maps = bundle["context_cat_maps"]
context_cat_encoder.cardinalities = {
    col: len(vocab) for col, vocab in context_cat_encoder.maps.items()
}

wagon_num_scaler = NumericScaler()
wagon_num_scaler.cols = wagon_num_cols
wagon_num_scaler.means = pd.Series(bundle["wagon_num_means"])
wagon_num_scaler.stds = pd.Series(bundle["wagon_num_stds"])

context_num_scaler = NumericScaler()
context_num_scaler.cols = context_num_cols
context_num_scaler.means = pd.Series(bundle["context_num_means"])
context_num_scaler.stds = pd.Series(bundle["context_num_stds"])

model = TwoTowerNetV2(
    wagon_cat_cardinalities=wagon_cat_cardinalities,
    wagon_num_features=len(wagon_num_cols),
    context_cat_cardinalities=context_cat_cardinalities,
    context_num_features=len(context_num_cols),
    wagon_hidden=(256, 128),
    context_hidden=(128, 64),
    fusion_hidden=(256, 128, 64),
    dropout=0.20
).to(DEVICE)

model.load_state_dict(bundle["model_state_dict"])
model.eval()


# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(DATA_PATH)

for col in TARGET_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

all_needed_cols = set(wagon_cat_cols + wagon_num_cols + context_cat_cols + context_num_cols)
for col in all_needed_cols:
    if col not in df.columns:
        df[col] = np.nan

id_col = "wagon_id"
date_col = first_existing(DATE_CANDIDATES, df)

if id_col not in df.columns:
    raise ValueError(f"Column '{id_col}' not found in dataset")

print("Rows before dedup:", len(df))
print("Unique wagon_id before dedup:", df[id_col].nunique())
print("Detected date column:", date_col)

if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values([id_col, date_col])
    df = df.drop_duplicates(subset=[id_col], keep="last").copy()
else:
    df = df.drop_duplicates(subset=[id_col], keep="last").copy()

df = df.reset_index(drop=True)

print("Rows after dedup:", len(df))
print("Unique wagon_id after dedup:", df[id_col].nunique())


# =========================================================
# PREPROCESS
# =========================================================

wagon_cat = wagon_cat_encoder.transform(df, wagon_cat_cols) if wagon_cat_cols else pd.DataFrame(index=df.index)
wagon_num = wagon_num_scaler.transform(df) if wagon_num_cols else pd.DataFrame(index=df.index)
context_cat = context_cat_encoder.transform(df, context_cat_cols) if context_cat_cols else pd.DataFrame(index=df.index)
context_num = context_num_scaler.transform(df) if context_num_cols else pd.DataFrame(index=df.index)

wagon_cat_tensor = torch.tensor(wagon_cat.values, dtype=torch.long, device=DEVICE) if wagon_cat.shape[1] > 0 else torch.empty((len(df), 0), dtype=torch.long, device=DEVICE)
wagon_num_tensor = torch.tensor(wagon_num.values, dtype=torch.float32, device=DEVICE) if wagon_num.shape[1] > 0 else torch.empty((len(df), 0), dtype=torch.float32, device=DEVICE)
context_cat_tensor = torch.tensor(context_cat.values, dtype=torch.long, device=DEVICE) if context_cat.shape[1] > 0 else torch.empty((len(df), 0), dtype=torch.long, device=DEVICE)
context_num_tensor = torch.tensor(context_num.values, dtype=torch.float32, device=DEVICE) if context_num.shape[1] > 0 else torch.empty((len(df), 0), dtype=torch.float32, device=DEVICE)


# =========================================================
# PREDICT
# =========================================================

with torch.no_grad():
    logit_14, logit_90 = model(
        wagon_cat_tensor,
        wagon_num_tensor,
        context_cat_tensor,
        context_num_tensor
    )
    p14 = torch.sigmoid(logit_14).cpu().numpy().ravel()
    p90 = torch.sigmoid(logit_90).cpu().numpy().ravel()

pred14 = (p14 >= best_threshold_14).astype(int)
pred90 = (p90 >= best_threshold_90).astype(int)


# =========================================================
# SHAP
# =========================================================

flat_parts = []
feature_names = []

if wagon_cat.shape[1] > 0:
    tmp = wagon_cat.copy()
    tmp.columns = [f"wagon_cat::{c}" for c in tmp.columns]
    flat_parts.append(tmp)
    feature_names.extend(tmp.columns.tolist())

if wagon_num.shape[1] > 0:
    tmp = wagon_num.copy()
    tmp.columns = [f"wagon_num::{c}" for c in tmp.columns]
    flat_parts.append(tmp)
    feature_names.extend(tmp.columns.tolist())

if context_cat.shape[1] > 0:
    tmp = context_cat.copy()
    tmp.columns = [f"context_cat::{c}" for c in tmp.columns]
    flat_parts.append(tmp)
    feature_names.extend(tmp.columns.tolist())

if context_num.shape[1] > 0:
    tmp = context_num.copy()
    tmp.columns = [f"context_num::{c}" for c in tmp.columns]
    flat_parts.append(tmp)
    feature_names.extend(tmp.columns.tolist())

flat_df = pd.concat(flat_parts, axis=1)

n_wagon_cat = wagon_cat.shape[1]
n_wagon_num = wagon_num.shape[1]
n_context_cat = context_cat.shape[1]
n_context_num = context_num.shape[1]

wagon_cat_cards = wagon_cat_cardinalities
context_cat_cards = context_cat_cardinalities


def flat_to_model_inputs(X):
    X = np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(1, -1)

    pos = 0

    if n_wagon_cat > 0:
        x_wc = np.rint(X[:, pos:pos+n_wagon_cat]).astype(np.int64)
        for j, card in enumerate(wagon_cat_cards):
            x_wc[:, j] = np.clip(x_wc[:, j], 0, card - 1)
        pos += n_wagon_cat
        x_wc = torch.tensor(x_wc, dtype=torch.long, device=DEVICE)
    else:
        x_wc = torch.empty((len(X), 0), dtype=torch.long, device=DEVICE)

    if n_wagon_num > 0:
        x_wn = X[:, pos:pos+n_wagon_num].astype(np.float32)
        pos += n_wagon_num
        x_wn = torch.tensor(x_wn, dtype=torch.float32, device=DEVICE)
    else:
        x_wn = torch.empty((len(X), 0), dtype=torch.float32, device=DEVICE)

    if n_context_cat > 0:
        x_cc = np.rint(X[:, pos:pos+n_context_cat]).astype(np.int64)
        for j, card in enumerate(context_cat_cards):
            x_cc[:, j] = np.clip(x_cc[:, j], 0, card - 1)
        pos += n_context_cat
        x_cc = torch.tensor(x_cc, dtype=torch.long, device=DEVICE)
    else:
        x_cc = torch.empty((len(X), 0), dtype=torch.long, device=DEVICE)

    if n_context_num > 0:
        x_cn = X[:, pos:pos+n_context_num].astype(np.float32)
        pos += n_context_num
        x_cn = torch.tensor(x_cn, dtype=torch.float32, device=DEVICE)
    else:
        x_cn = torch.empty((len(X), 0), dtype=torch.float32, device=DEVICE)

    return x_wc, x_wn, x_cc, x_cn


def predict_prob_14(X):
    x_wc, x_wn, x_cc, x_cn = flat_to_model_inputs(X)
    with torch.no_grad():
        logit_14, _ = model(x_wc, x_wn, x_cc, x_cn)
        return torch.sigmoid(logit_14).cpu().numpy().ravel()


def predict_prob_90(X):
    x_wc, x_wn, x_cc, x_cn = flat_to_model_inputs(X)
    with torch.no_grad():
        _, logit_90 = model(x_wc, x_wn, x_cc, x_cn)
        return torch.sigmoid(logit_90).cpu().numpy().ravel()


background_size = min(BACKGROUND_SIZE, len(flat_df))

if EXPLAIN_MAX_ROWS is None:
    EXPLAIN_SIZE = len(flat_df)
else:
    EXPLAIN_SIZE = min(len(flat_df), EXPLAIN_MAX_ROWS)

print("SHAP background size:", background_size)
print("SHAP explain size:", EXPLAIN_SIZE)

background = shap.sample(flat_df, background_size, random_state=SEED)
explain_df = flat_df.iloc[:EXPLAIN_SIZE].copy()

explainer_14 = shap.KernelExplainer(predict_prob_14, background.values)
explainer_90 = shap.KernelExplainer(predict_prob_90, background.values)

shap_values_14 = np.array(explainer_14.shap_values(explain_df.values, nsamples=100))
shap_values_90 = np.array(explainer_90.shap_values(explain_df.values, nsamples=100))


def prettify_feature_name(name):
    name = str(name)
    name = name.replace("wagon_cat::", "")
    name = name.replace("wagon_num::", "")
    name = name.replace("context_cat::", "")
    name = name.replace("context_num::", "")
    return name


def build_top_shap_items(shap_row, x_row, feature_names, top_k=5, positive_only=True):
    items = []

    for fname, shap_val, x_val in zip(feature_names, shap_row, x_row):
        items.append({
            "feature": prettify_feature_name(fname),
            "value": float(shap_val),
            "prettyValue": str(x_val)
        })

    if positive_only:
        items = [x for x in items if x["value"] > 0]
        items = sorted(items, key=lambda z: z["value"], reverse=True)
    else:
        items = sorted(items, key=lambda z: abs(z["value"]), reverse=True)

    return items[:top_k]


# =========================================================
# SIMPLE ATTACHED FACTOR LAYER FOR UI
# =========================================================

all_model_cols = wagon_cat_cols + wagon_num_cols + context_cat_cols + context_num_cols

feature_groups = {
    "repair_history": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["repair", "ремонт", "неисправ", "nrp", "замен", "окончание"])
    ],
    "operational_load": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["distance", "пробег", "рейс", "shipment", "count_", "total_", "days_"])
    ],
    "weather": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["weather", "temp", "температур", "rain", "snow", "wind", "дожд", "снег", "ветер"])
    ],
    "route_station_context": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["отправ", "назнач", "station", "route", "маршрут", "стр.", "станц", "город", "страна", "country"])
    ],
    "composition_context": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["train_id", "composition", "month", "месяц"])
    ],
    "wagon_intrinsic": [
        c for c in all_model_cols
        if any(k in c.lower() for k in ["вагон", "wagon", "возраст", "age", "build", "построй", "колес", "рама", "балка"])
    ]
}
feature_groups = {k: v for k, v in feature_groups.items() if len(v) > 0}


def attached_drivers_from_row(row):
    drivers = []

    for col in row.index:
        val = row[col]
        col_low = str(col).lower()

        if pd.isna(val):
            continue

        if any(k in col_low for k in ["temp", "температур"]):
            drivers.append({"feature": "Температура", "value": str(val)})
        elif any(k in col_low for k in ["wind", "ветер"]):
            drivers.append({"feature": "Скорость ветра", "value": str(val)})
        elif any(k in col_low for k in ["rain", "дожд"]):
            drivers.append({"feature": "Осадки", "value": str(val)})
        elif any(k in col_low for k in ["snow", "снег"]):
            drivers.append({"feature": "Снег", "value": str(val)})
        elif any(k in col_low for k in ["repair", "ремонт"]):
            drivers.append({"feature": col, "value": str(val)})
        elif any(k in col_low for k in ["distance", "пробег"]):
            drivers.append({"feature": col, "value": str(val)})
        elif any(k in col_low for k in ["отправ", "station", "route", "маршрут"]):
            drivers.append({"feature": col, "value": str(val)})
        elif any(k in col_low for k in ["age", "возраст", "build", "построй"]):
            drivers.append({"feature": col, "value": str(val)})

    seen = set()
    out = []
    for d in drivers:
        key = (d["feature"], d["value"])
        if key not in seen:
            seen.add(key)
            out.append(d)

    return out[:5]


def attached_group_drivers_from_row(row):
    groups = []
    cols_present = {c for c in row.index if not pd.isna(row[c])}

    for group_name, cols in feature_groups.items():
        if any(c in cols_present for c in cols):
            groups.append({"group": group_name.replace("_", " ").title()})

    return groups[:4]


# =========================================================
# MORE COLUMN PICKERS
# =========================================================

location_col = first_existing([
    "Ст. отправления",
    "Станция отправления",
    "location",
    "current_location",
    "station_from"
], df)

destination_col = first_existing([
    "Ст. назначения",
    "Станция назначения",
    "destination",
    "station_to"
], df)

country_from_col = "Стр. отправления"

country_to_col = first_existing([
    "Страна назначения",
    "Страна назн.",
    "Country to",
    "country_to"
], df)

weather_col = first_existing([
    "weather", "Погода", "Weather condition"
], df)

temp_col = first_existing([
    "Средняя температура (°C)", "temp", "temperature"
], df)

wind_col = first_existing([
    "Средняя скорость ветра (м/с)", "wind", "wind_speed"
], df)

# правило: все станции с припиской (эксп) считаем Россией
if location_col is not None and country_from_col is not None:
    mask_exp = df[location_col].apply(is_russia_exp_station)
    df.loc[mask_exp, country_from_col] = "РОССИЯ"
    print("Станций с (эксп), принудительно отнесенных к России:", int(mask_exp.sum()))


# =========================================================
# BUILD UI JSON
# =========================================================

wagons = []

for i, row in df.iterrows():
    wagon_id = safe_value(row[id_col]) if id_col else f"WGN-{i+1:04d}"
    location = safe_value(row[location_col]) if location_col else "Unknown location"
    destination = safe_value(row[destination_col]) if destination_col else "Unknown destination"
    route = f"{location} → {destination}"

    country_from = safe_value(row[country_from_col]) if country_from_col else None
    country_to = safe_value(row[country_to_col]) if country_to_col else None

    weather = safe_value(row[weather_col]) if weather_col else None
    temperature = safe_float(row[temp_col]) if temp_col else None
    wind_speed = safe_float(row[wind_col]) if wind_col else None

    profile_cols = []
    for c in df.columns:
        cl = str(c).lower()
        if any(k in cl for k in [
            "weight", "distance", "пробег", "days", "year",
            "shipments", "рейс", "возраст", "age", "build"
        ]):
            profile_cols.append(c)

    profile = {}
    for c in profile_cols[:8]:
        profile[c] = safe_value(row[c])

    shap_top_14 = build_top_shap_items(
        shap_values_14[i],
        explain_df.iloc[i].values,
        feature_names,
        top_k=5,
        positive_only=True
    )
    shap_top_90 = build_top_shap_items(
        shap_values_90[i],
        explain_df.iloc[i].values,
        feature_names,
        top_k=5,
        positive_only=True
    )

    wagons.append({
        "wagonId": str(wagon_id),
        "location": str(location),
        "destination": str(destination),
        "route": route,
        "countryFrom": country_from,
        "countryTo": country_to,
        "risk14d": round(float(p14[i]), 4),
        "risk90d": round(float(p90[i]), 4),
        "pred14d": int(pred14[i]),
        "pred90d": int(pred90[i]),
        "riskLevel": risk_level(float(p14[i]), float(p90[i]), best_threshold_14, best_threshold_90),
        "weather": weather,
        "temperature": temperature,
        "windSpeed": wind_speed,
        "profile": profile,
        "drivers14d": attached_drivers_from_row(row),
        "drivers90d": attached_drivers_from_row(row),
        "groupDrivers14d": attached_group_drivers_from_row(row),
        "groupDrivers90d": attached_group_drivers_from_row(row),
        "shap14d": shap_top_14,
        "shap90d": shap_top_90,
    })

summary = {
    "totalWagons": int(df[id_col].nunique()),
    "locations": len(set(w["location"] for w in wagons)),
    "highRisk14d": int(sum(w["pred14d"] for w in wagons)),
    "highRisk90d": int(sum(w["pred90d"] for w in wagons)),
    "threshold14d": float(best_threshold_14),
    "threshold90d": float(best_threshold_90),
    "modelName": "Two-Tower Wagon Failure Risk Model",
    "notes": "Predictions are real outputs from the uploaded trained PyTorch model on the latest available record for each wagon. SHAP is calculated for all wagons."
}

result = {
    "summary": summary,
    "wagons": wagons
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    check = json.load(f)

print(f"Saved to {OUTPUT_PATH}")
print(f"Wagons in JSON: {len(check['wagons'])}")
print(f"Unique wagons: {check['summary']['totalWagons']}")
print(f"Locations: {check['summary']['locations']}")
print(f"High risk 14d: {check['summary']['highRisk14d']}")
print(f"High risk 90d: {check['summary']['highRisk90d']}")

cnt14 = sum(1 for w in check["wagons"] if len(w.get("shap14d", [])) > 0)
cnt90 = sum(1 for w in check["wagons"] if len(w.get("shap90d", [])) > 0)

print("Wagons with shap14d:", cnt14)
print("Wagons with shap90d:", cnt90)

Rows before dedup: 761294
Unique wagon_id before dedup: 5127
Detected date column: None
Rows after dedup: 5127
Unique wagon_id after dedup: 5127
SHAP background size: 80
SHAP explain size: 5127


  0%|          | 0/5127 [00:00<?, ?it/s]

  0%|          | 0/5127 [00:00<?, ?it/s]

Станций с (эксп), принудительно отнесенных к России: 715
Saved to wagons_data.json
Wagons in JSON: 5127
Unique wagons: 5127
Locations: 466
High risk 14d: 3220
High risk 90d: 2968
Wagons with shap14d: 4540
Wagons with shap90d: 4498
